# ARIMA（自回歸整合移動平均，1970）

- **核心原理**：自回歸＋移動平均＋差分
- **變體**：SARIMA（季節性）、ARIMAX（外生變數）、Auto-ARIMA（自動選階數）
- **推薦資料集**：AirPassengers（同第 1 章：train 1949–1958、test 1959–1960）
- **實作**：✋ 手刻 AR 部分（OLS、Yule-Walker、遞迴預測、差分還原）；完整 ARIMA／SARIMA 用 🧰 statsmodels
- **文獻**：Box & Jenkins (1970), *Time Series Analysis: Forecasting and Control*；fpp3 第 9 章

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

和第 1 章（指數平滑）的差別：

| | 第 1 章 指數平滑 | 本章 ARIMA |
| --- | --- | --- |
| 想法 | 把序列拆成水準、趨勢、季節，**各自平滑** | 先**差分**到平穩，再用「過去的值」（AR）與「過去的誤差」（MA）做**回歸** |
| 處理趨勢／季節 | 用狀態 $b_t$、$s_t$ 追蹤 | 差分：$y_t - y_{t-1}$（去趨勢）、$y_t - y_{t-12}$（去季節） |
| 參數估計 | 最小化一步誤差的 SSE | AR：OLS／Yule-Walker（本章手寫）；含 MA：最大概似（statsmodels） |
| 選模型 | 選元件（趨勢、季節、阻尼） | 選階數 $(p, d, q)(P, D, Q)_m$：看 ACF／PACF、AIC |

一句話：**ARIMA 把「預測 $y$」轉成「預測差分後的序列」—— 差分之後的序列平穩（平均、變異、自我相關都不隨時間改變），可以用一個固定係數的線性回歸（對自己的過去）來預測，最後再把差分「加回去」。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3E、4、5 節開頭另有各自的符號表或名詞表。

**資料**：同第 1 章 —— `load_air_passengers()`、`split_series(y, test_size=24)`；120 個月訓練、24 個月測試。本章大部分在 **log 尺度**上建模（第 1 章實驗 3：取 log 讓季節波動變成固定大小），最後 exp 回來。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $y_t$、$x_t = \log y_t$ | 原始序列、取 log | |
| $\nabla x_t = x_t - x_{t-1}$ | 一階差分（$d = 1$） | |
| $\nabla_m x_t = x_t - x_{t-m}$ | 季節差分（$D = 1$，$m = 12$） | |
| $z_t = \nabla\nabla_{12} x_t$ | 兩次差分後的序列（本章手寫 AR 的對象） | 107 個值 |
| $p$、$q$ | AR、MA 的階數（用幾個過去的值／誤差） | |
| $(P, D, Q)_m$ | 季節部分的階數 | |
| $\phi_k$、$\theta_k$ | AR、MA 係數 | |
| $c$ | 常數項 | |
| $\varepsilon_t$ | 白雜訊（平均 0、變異 $\sigma^2$、彼此不相關） | |
| $r_k$ | 樣本自我相關（ACF） | |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第 1 章 | 指數平滑（`06-time-series/01-exponential-smoothing`） | 季節性 naive test MASE 2.49；CV 選出「log + Holt-Winters 加法」（CV MASE 0.61、test MASE 0.66）；ETS(M,A,M) AIC 最低但 test MASE 1.69；滾動起點 CV（起點 84、96、108，各預測 12 個月）。成績在 `results/air_passengers.csv`。 |
| 第 1 章 EDA | 同上 | 原始序列 ACF 每個 lag 都很高（趨勢）；一階差分後 lag 12 最突出（0.80）；季節波動隨水準變大、log 之後大致固定。 |
| 第一部分第 2 章 | 線性回歸 | OLS：$\hat\beta = \arg\min\lVert y - X\beta\rVert^2$，`np.linalg.lstsq(X, y)`；AR 就是「拿自己的過去當特徵」的線性回歸。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 平穩（weakly stationary）**：平均 $E[z_t]$、變異 $\text{Var}(z_t)$、自我共變異 $\text{Cov}(z_t, z_{t-k})$ 都**不隨 $t$ 改變**（只跟 lag $k$ 有關）。有趨勢 → 平均會變 → 不平穩；有季節 → 平均隨月份變 → 不平穩。

**2. 差分**：$\nabla x_t = x_t - x_{t-1}$ 去掉線性趨勢（例：$x = (1, 3, 5, 7)$ → $\nabla x = (2, 2, 2)$）；$\nabla_{12} x_t = x_t - x_{t-12}$ 去掉固定的季節形狀。**還原**：$x_t = \nabla x_t + x_{t-1}$（一階）；兩次差分 $z_t = (x_t - x_{t-1}) - (x_{t-12} - x_{t-13})$ → $x_t = z_t + x_{t-1} + x_{t-12} - x_{t-13}$。

**3. AR(p)**：$z_t = c + \phi_1 z_{t-1} + \cdots + \phi_p z_{t-p} + \varepsilon_t$ —— 用自己過去 $p$ 期的線性組合預測。AR(1) 的例子：$\phi = 0.5$、$z_{t-1} = 4$、$c = 0$ → $\hat z_t = 2$。

**4. MA(q)**：$z_t = c + \varepsilon_t + \theta_1\varepsilon_{t-1} + \cdots + \theta_q\varepsilon_{t-q}$ —— 用過去的「預測誤差」修正。$\varepsilon$ 看不到，要邊算邊估 → 沒有 OLS 閉式解，用最大概似（數值最佳化）。

**5. 自我相關 ACF 與偏自我相關 PACF**：$r_k$ = $z_t$ 與 $z_{t-k}$ 的相關係數（第 1 章的 `acf`）；PACF 的第 $k$ 個值 = 「控制了 lag $1..k-1$ 之後」$z_{t-k}$ 還多提供多少 = **AR($k$) 回歸裡 $z_{t-k}$ 的係數**。經驗法則：AR($p$) 的 PACF 在 lag $p$ 之後截斷；MA($q$) 的 ACF 在 lag $q$ 之後截斷。

**6. Yule-Walker 方程**：AR($p$) 的係數滿足 $r_k = \sum_{j=1}^p \phi_j r_{|k-j|}$（$k = 1..p$）—— 一個 $p \times p$ 的線性方程組 $R\phi = r$，$R_{ij} = r_{|i-j|}$（Toeplitz 矩陣）。例：AR(1) → $\phi_1 = r_1$。

**7. 平穩條件**：AR 的特徵多項式 $1 - \phi_1 u - \cdots - \phi_p u^p = 0$ 的根都在**單位圓外**（$|u| > 1$）。AR(1)：$|\phi_1| < 1$。

**8. AIC**：$-2\log L + 2k$（$k$ = 參數個數）；越小越好；在「配適好」與「參數少」之間取捨。

#### 程式

**9. `np.column_stack`**：把幾條長度一樣的向量並排成矩陣的欄。**10. `np.linalg.lstsq(X, y, rcond=None)[0]`**：OLS 係數。**11. `np.linalg.solve(A, b)`**：解 $A x = b$。**12. `np.roots(coeffs)`**：多項式的根（係數從最高次排到常數）。

---
## 1. 資料與探索性資料分析（EDA）：差分到平穩

第 1 章已經看過趨勢與季節（見跨章引用）。這一章的 EDA 只做一件事：**找到讓序列平穩的轉換**。工具：

- 畫圖 + ACF（第 1 章的 `acf`，這裡直接搬過來）；
- **ADF 檢定**（Augmented Dickey-Fuller，`statsmodels.tsa.stattools.adfuller`）：虛無假設「有單位根（不平穩，像隨機漫步）」。p 值小（< 0.05）→ 拒絕 → 平穩。

In [ ]:
import itertools
import time
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from ml_dl_llm.data import load_air_passengers, split_series

warnings.filterwarnings("ignore", category=UserWarning)  # statsmodels 的收斂／頻率提醒
warnings.filterwarnings("ignore", category=FutureWarning)  # statsmodels 未來版本的回傳格式提醒
y = load_air_passengers()
if isinstance(y.index, pd.PeriodIndex):
    y.index = y.index.to_timestamp()
y = y.astype(float).rename("passengers")
y_train, y_test = split_series(y, test_size=24)
M, H = 12, len(y_test)
x_train = np.log(y_train)


def acf(x: np.ndarray, max_lag: int) -> np.ndarray:
    """第 1 章的樣本自我相關 r_k，k = 0..max_lag。"""
    x = np.asarray(x, dtype=float) - np.mean(x)
    denom = (x**2).sum()
    return np.array([(x[k:] * x[: len(x) - k]).sum() / denom for k in range(max_lag + 1)])


transforms = {
    "y": y_train,
    "log y": x_train,
    "∇ log y": x_train.diff(),
    "∇12 log y": x_train.diff(M),
    "∇∇12 log y": x_train.diff().diff(M),
}
fig, axes = plt.subplots(len(transforms), 2, figsize=(15, 12))
rows = []
for (name, s), (ax1, ax2) in zip(transforms.items(), axes):
    s = s.dropna()
    s.plot(ax=ax1)
    ax1.set_title(name, fontsize=10)
    r = acf(s.to_numpy(), 36)
    ax2.bar(np.arange(37), r)
    ax2.axhline(2 / np.sqrt(len(s)), color="gray", ls=":")
    ax2.axhline(-2 / np.sqrt(len(s)), color="gray", ls=":")
    adf = adfuller(s.to_numpy())
    rows.append({"series": name, "n": len(s), "ADF stat": adf[0], "ADF p-value": adf[1], "r_1": r[1], "r_12": r[12]})
plt.tight_layout()
plt.show()
display(pd.DataFrame(rows).set_index("series").round(4))

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 讀表格：哪幾個轉換的 ADF p 值 < 0.05？只做一階差分（∇ log y）夠嗎？它的 ACF 在 lag 12、24 還有什麼？
# 答：
#
# (2) ∇∇12 log y（兩次差分）：
#     (a) 它的圖還有沒有趨勢、季節？ACF 還有哪些 lag 超出虛線（±2/√n）？
#     (b) 兩次差分之後少了幾個資料點？為什麼？
#     (c) 這條序列的「意義」是什麼？（提示：∇12 log y ≈ 年成長率；再差分一次 = 年成長率的「變化」）
# 答：
#
# (3) 為什麼要先取 log 再差分？如果直接對 y 差分，差分後的變異會隨時間怎樣？（第 1 章 EDA：季節波動隨水準變大）
# 答：

---
## 2. Baseline

| Baseline | 說明 |
| --- | --- |
| 季節性 naive | 第 1 章：test MASE 2.49 |
| **「差分後預測 0」** | $\hat z = 0$ → $x_{n+h}$ 用差分還原（3D 的 TODO 7）：「明年的形狀 = 今年的形狀 + 去年到今年的變化」—— 這其實就是 ARIMA(0,1,0)(0,1,0)$_{12}$（兩次差分的隨機漫步） |
| 第 1 章最好的模型 | log + Holt-Winters 加法：test MASE 0.66（在第 6 節用 statsmodels 重算，放進同一張表） |

（「差分後預測 0」要用到 TODO 7，放在 3D 之後算。）

In [ ]:
from ml_dl_llm.metrics import forecast_metrics

snaive = y_train.to_numpy()[len(y_train) - M + (np.arange(H) % M)]
test_table = pd.DataFrame({"Seasonal naive": forecast_metrics(y_test, snaive, y_train, season=M)}).T
display(test_table)

In [ ]:
# TODO 2（問答）先寫下預測（第 3D、6 節驗證）
#
# (1) 「差分後預測 0」（兩次差分的隨機漫步）跟季節性 naive 比，誰比較準？為什麼？（提示：季節性 naive 假設「明年 = 今年」；兩次差分的隨機漫步假設「明年 = 今年 + 今年比去年多的部分」）
# (2) 用 AR 去預測 z（差分後的序列）能比「預測 0」好多少？（提示：z 的自我相關有多強？看 EDA 的 r_1、r_12）
# 答：

---
## 3. 手寫模型：AR($p$) 與差分還原

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 把序列變成「特徵矩陣」：lag 矩陣 | 3 |
| 3B | AR 的兩種估計：OLS、Yule-Walker | 4、5 |
| 3C | 遞迴預測 $h$ 步 | 6 |
| 3D | 差分還原：從 $\hat z$ 回到 $\hat y$；TODO 3–7 的問答 | 7、8 |
| 3E | 實驗：PACF、選 $p$（AIC）、常數項、平穩條件 | 9 |

**程式結構**：`ARScratch(p, const)` 仿 sklearn —— `fit(z)` 估計 `coef_`（$\phi_1..\phi_p$）、`intercept_`（$c$）、`sigma2_`；`forecast(h)` 回傳 $\hat z_{n+1..n+h}$。差分與還原是獨立的函式（`difference`、`undifference`）。

### 3A. lag 矩陣：AR 就是線性回歸

$$\underbrace{\begin{pmatrix} z_{p+1} \\ z_{p+2} \\ \vdots \\ z_n \end{pmatrix}}_{\text{target}\ (n-p,)} \approx \underbrace{\begin{pmatrix} z_p & z_{p-1} & \cdots & z_1 \\ z_{p+1} & z_p & \cdots & z_2 \\ \vdots & & & \vdots \\ z_{n-1} & z_{n-2} & \cdots & z_{n-p}\end{pmatrix}}_{X\ (n-p,\ p)\text{：第 } k \text{ 欄 = lag } k} \begin{pmatrix}\phi_1 \\ \vdots \\ \phi_p\end{pmatrix} + c$$

#### 符號表（3A–3C）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $z$ | 要建模的平穩序列（0 起算） | $(n,)$ | `z` |
| $X$ | lag 矩陣：`X[i, k-1]` = $z_{p+i-k}$ | $(n - p, p)$ | `X` |
| target | $z_{p..n-1}$ | $(n - p,)$ | `target` |
| $\phi$、$c$ | 係數、常數 | $(p,)$、純量 | `coef_`、`intercept_` |

**下標公式**：
- TODO 3：第 $k$ 欄（lag $k$，$k = 1..p$）= `z[p - k : n - k]`；`target = z[p:]`。

#### 最小例子：TODO 3

$z = (1, 2, 3, 4, 5)$、$p = 2$。

1. **正確答案**（迴圈）：$X = \begin{pmatrix}2 & 1\\3 & 2\\4 & 3\end{pmatrix}$（每列是「前 1 期、前 2 期」）、target $= (3, 4, 5)$。
2. **積木**：`z[p - k : n - k]` 對 $k = 1$、$k = 2$ 各是什麼。
3. **留給你**：`np.column_stack`。

In [ ]:
# 最小例子：TODO 3
z_mini = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
p_mini = 2
# ---- 1. 正確答案：迴圈 ----
rows_mini = [[z_mini[t - k] for k in range(1, p_mini + 1)] for t in range(p_mini, len(z_mini))]
print("X =", rows_mini, "；target =", z_mini[p_mini:])
# ---- 2. 積木 ----
for k in (1, 2):
    print(f"lag {k}：z[{p_mini} − {k} : {len(z_mini)} − {k}] =", z_mini[p_mini - k : len(z_mini) - k])
# ---- 3. 留給你 ----
# 寫完 TODO 3、執行過下兩格之後，取消下面的註解來對照：
# print(lag_matrix(z_mini, 2))

### 3B. AR 的兩種估計

| 方法 | 做法 | 性質 |
| --- | --- | --- |
| **OLS**（條件最小平方） | 對 lag 矩陣做線性回歸（第一部分第 2 章） | = statsmodels `AutoReg`；不保證估出來的模型平穩 |
| **Yule-Walker** | 用樣本 ACF 解 $R\phi = r$（前置複習 6） | 估出來的模型一定平穩；有偏誤（小樣本時係數偏向 0） |

兩者在大樣本時接近。Yule-Walker 沒有常數項（先減掉平均）。

**下標公式**：
- TODO 4：有常數時 `X1 = np.column_stack([np.ones(len(X)), X])`，`beta = np.linalg.lstsq(X1, target, rcond=None)[0]` → `intercept_ = beta[0]`、`coef_ = beta[1:]`；沒有常數時直接用 `X`、`intercept_ = 0`。
- TODO 5：`r = acf(z, p)`；$R_{ij} = r_{|i-j|}$（$i, j = 0..p-1$）；`np.linalg.solve(R, r[1:])`。

#### 最小例子：TODO 4、5

- TODO 4：$z = (8, 4, 2, 1, 0.5)$，每一期剛好是前一期的一半 → AR(1)、沒有常數：OLS 的 $\phi_1$ 應該**剛好** = 0.5。
- TODO 5：如果 ACF 是 $r = (1, 0.5, 0.25)$（AR(1)、$\phi = 0.5$ 的理論 ACF），Yule-Walker 解 AR(2)：$\begin{pmatrix}1 & 0.5\\0.5 & 1\end{pmatrix}\phi = \begin{pmatrix}0.5\\0.25\end{pmatrix}$ → $\phi = (0.5, 0)$（第 2 個係數是 0：AR(1) 的 PACF 在 lag 1 之後截斷）。

In [ ]:
from typing import Self


def lag_matrix(z: np.ndarray, p: int) -> tuple[np.ndarray, np.ndarray]:
    """回傳 (X (n − p, p)，第 k 欄 = lag k；target (n − p,))。"""
    z = np.asarray(z, dtype=float)
    n = len(z)
    # TODO 3（程式）：見下標公式
    raise NotImplementedError("TODO 3：lag 矩陣")
    return X, target


class ARScratch:
    """AR(p)：OLS 或 Yule-Walker。屬性：coef_ (p,)、intercept_、sigma2_、resid_、aic_。"""

    def __init__(self, p: int, const: bool = True, method: str = "ols") -> None:
        self.p, self.const, self.method = p, const, method

    def fit(self, z: np.ndarray) -> Self:
        z = np.asarray(z, dtype=float)
        self.z_ = z
        X, target = lag_matrix(z, self.p)
        if self.method == "ols":
            # TODO 4（程式）：OLS（有／沒有常數項）
            raise NotImplementedError("TODO 4：OLS")
        else:
            mu = z.mean() if self.const else 0.0
            r = acf(z - mu, self.p) if self.const else acf(z, self.p)
            # TODO 5（程式）：Yule-Walker（R 是 Toeplitz：R[i, j] = r[|i − j|]）
            raise NotImplementedError("TODO 5：Yule-Walker")
            self.intercept_ = float(mu * (1 - self.coef_.sum()))  # 平均 μ 的 AR：c = μ(1 − Σφ)
        self.resid_ = target - (self.intercept_ + X @ self.coef_)
        n_eff = len(target)
        self.sigma2_ = float((self.resid_**2).mean())
        k = self.p + int(self.const) + 1  # 係數 + 常數 + σ²
        self.aic_ = n_eff * np.log(self.sigma2_) + 2 * k  # 高斯概似（省略常數項）
        return self

    def forecast(self, h: int) -> np.ndarray:
        """遞迴的 h 步預測 ẑ_{n+1..n+h}。"""
        hist = list(self.z_[-self.p :])  # 最後 p 個值（舊 → 新）
        out = []
        for _ in range(h):
            # TODO 6（程式）：z_next = c + Σ_k φ_k · hist[−k]（k = 1..p）；append 到 hist 與 out
            #   常見錯誤：hist[−1] 是 lag 1（最新的），coef_[0] 是 φ_1 —— 要配對 coef_[k − 1] 與 hist[−k]
            raise NotImplementedError("TODO 6：遞迴預測")
        return np.array(out)


# 最小例子的對照（寫完 TODO 3–6 之後執行）
# print(ARScratch(1, const=False).fit(np.array([8.0, 4.0, 2.0, 1.0, 0.5])).coef_)                  # 應為 [0.5]
# print(ARScratch(1, const=False).fit(np.array([8.0, 4.0, 2.0, 1.0, 0.5])).forecast(2))            # 應為 [0.25 0.125]

In [ ]:
# Yule-Walker 的最小例子：直接解 R φ = r（不用資料）
r_mini = np.array([1.0, 0.5, 0.25])
R_mini = np.array([[r_mini[abs(i - j)] for j in range(2)] for i in range(2)])
print("R =\n", R_mini, "\nφ =", np.linalg.solve(R_mini, r_mini[1:]))

# 對照 statsmodels：AutoReg（OLS）、yule_walker、pacf
from statsmodels.regression.linear_model import yule_walker
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import pacf

z = x_train.diff().diff(M).dropna().to_numpy()
for p in [1, 3, 13]:
    ols = ARScratch(p).fit(z)
    sm_ar = AutoReg(z, lags=p, trend="c").fit()
    yw = ARScratch(p, method="yw").fit(z)
    sm_yw, _ = yule_walker(z, order=p, method="mle", demean=True)
    print(f"p = {p:2d}：✅ OLS vs AutoReg 係數差 {np.abs(np.r_[ols.intercept_, ols.coef_] - sm_ar.params).max():.1e}；"
          f"✅ Yule-Walker vs statsmodels {np.abs(yw.coef_ - sm_yw).max():.1e}；"
          f"✅ 遞迴預測 vs AutoReg {np.abs(ols.forecast(24) - sm_ar.forecast(24)).max():.1e}")
pacf_mine = np.array([ARScratch(k).fit(z).coef_[-1] for k in range(1, 25)])
print("✅ PACF（AR(k) 的最後一個係數）vs statsmodels pacf(method='ols')：", f"{np.abs(pacf_mine - pacf(z, nlags=24, method='ols')[1:]).max():.1e}")

### 3D. 差分還原：從 $\hat z$ 回到 $\hat y$

$z_t = (x_t - x_{t-1}) - (x_{t-m} - x_{t-m-1})$ → $x_t = z_t + x_{t-1} + x_{t-m} - x_{t-m-1}$（前置複習 2）。預測時一步一步往前推：每一步需要的 $x_{t-1}, x_{t-m}, x_{t-m-1}$ 可能是**已知的歷史**、也可能是**前面幾步剛預測出來的**（$h > 1$ 時）。

#### 符號表（3D）

| 符號 | 意義 | 程式裡 |
| --- | --- | --- |
| `x_hist` | 已知的 $x$（log y）的歷史 | `list(x_hist)`，一路 append 預測值 |
| `z_fc` | 預測的 $\hat z_{n+1..n+h}$ | $(h,)$ |
| 回傳 | $\hat x_{n+1..n+h}$ | $(h,)$ |

**下標公式**：
- TODO 7：`x_next = z_fc[k] + xs[-1] + xs[-m] - xs[-m - 1]`；`xs.append(x_next)`。

#### 最小例子：TODO 7（$m = 2$）

$x = (0, 1, 2, 3)$、$\hat z = (0, 0)$ → $\hat x_5 = 0 + 3 + 2 - 1 = 4$、$\hat x_6 = 0 + 4 + 3 - 2 = 5$（「差分後預測 0」就是延續最近的趨勢與季節形狀）。

In [ ]:
def undifference(x_hist: np.ndarray, z_fc: np.ndarray, m: int = M) -> np.ndarray:
    """把 ∇∇_m 的預測還原成 x 的預測。x_hist：已知的 x（至少 m + 1 個）。"""
    xs = list(np.asarray(x_hist, dtype=float))
    out = []
    for k in range(len(z_fc)):
        # TODO 7（程式）：x_next = z_fc[k] + xs[−1] + xs[−m] − xs[−m − 1]；append 到 xs、out
        raise NotImplementedError("TODO 7：差分還原")
    return np.array(out)


# 寫完 TODO 7 之後，取消下面的註解來對照（應為 [4. 5.]）：
# print(undifference(np.array([0.0, 1.0, 2.0, 3.0]), np.zeros(2), m=2))

# 檢查：把真實的 z 丟進去，應該還原出真實的 x
z_full = np.log(y).diff().diff(M).dropna().to_numpy()
x_rec = undifference(np.log(y).to_numpy()[: len(y_train)], z_full[-H:])
print("✅ 用真實的 z 還原 test 的 log y，最大差異：", f"{np.abs(x_rec - np.log(y_test).to_numpy()).max():.1e}")

fc_rw = np.exp(undifference(x_train.to_numpy(), np.zeros(H)))
test_table.loc["∇∇12 random walk (z = 0)"] = forecast_metrics(y_test, fc_rw, y_train, season=M)
display(test_table)

In [ ]:
# TODO 8（問答）TODO 3–7
#
# (1) 對照：OLS vs AutoReg、Yule-Walker vs statsmodels、PACF 的差異各是多少？
#     PACF 的定義是「AR(k) 回歸裡 z_{t−k} 的係數」—— 用 TODO 4 的 OLS 就能算，這跟「相關係數」有什麼不同？
# 答：
#
# (2) 差分還原：用真實的 z 還原的誤差是多少？「差分後預測 0」的 test MASE 是多少？比季節性 naive 好嗎？跟 TODO 2 (1) 的預測一樣嗎？
# 答：

### 3E. 實驗

| 實驗 | 問題 |
| --- | --- |
| 1 | ACF／PACF：$z$ 的 PACF 在哪些 lag 顯著？暗示什麼樣的 AR 階數？ |
| 2 | 選 $p$：AIC 隨 $p$ 怎麼變？有常數 vs 沒有常數，test 上差多少？ |
| 3 | 平穩條件：估出來的 AR 特徵多項式的根在單位圓外嗎？ |

In [ ]:
# 實驗 1：z 的 ACF 與 PACF
fig, axes = plt.subplots(1, 2, figsize=(15, 3.5))
band = 2 / np.sqrt(len(z))
axes[0].bar(np.arange(1, 37), acf(z, 36)[1:])
axes[1].bar(np.arange(1, 37), [ARScratch(k).fit(z).coef_[-1] for k in range(1, 37)])
for ax, t in zip(axes, ["ACF of z = ∇∇12 log y", "PACF of z (last coef of AR(k))"]):
    ax.axhline(band, color="gray", ls=":")
    ax.axhline(-band, color="gray", ls=":")
    ax.set_title(t)
    ax.set_xlabel("lag")
plt.show()
sig_acf = [k for k in range(1, 37) if abs(acf(z, 36)[k]) > band]
sig_pacf = [k for k in range(1, 37) if abs(ARScratch(k).fit(z).coef_[-1]) > band]
print("ACF 顯著的 lag：", sig_acf, "；PACF 顯著的 lag：", sig_pacf)

In [ ]:
# 實驗 2：選 p（AIC）；有／沒有常數項的 test 成績
# 注意：AR(p) 的 OLS 會丟掉前 p 個點 → 不同的 p 用的資料點數不一樣，AIC 不能直接比。
# 所以每個 p 都只用 z[P_MAX − p:]：所有模型的 target 都是同樣的 n − P_MAX 個點。
P_MAX = 14
rows = []
for p in [1, 2, 3, 6, 12, 13, 14]:
    for const in [True, False]:
        aic_common = ARScratch(p, const=const).fit(z[P_MAX - p :]).aic_
        mdl = ARScratch(p, const=const).fit(z)  # 預測用全部的資料重新估計
        fc = np.exp(undifference(x_train.to_numpy(), mdl.forecast(H)))
        rows.append({"p": p, "const": const, "AIC (train)": aic_common, "test MASE": forecast_metrics(y_test, fc, y_train, season=M)["mase"]})
exp2 = pd.DataFrame(rows)
display(exp2.pivot(index="p", columns="const", values=["AIC (train)", "test MASE"]).round(3))
p_aic = int(exp2[~exp2["const"]].sort_values("AIC (train)").iloc[0]["p"])
print("沒有常數項時 AIC 最低的 p：", p_aic)

In [ ]:
# 實驗 3：平穩條件 —— 特徵多項式 1 − φ1 u − … − φp u^p 的根
for p in [1, 3, p_aic]:
    mdl = ARScratch(p, const=False).fit(z)
    roots = np.roots(np.r_[-mdl.coef_[::-1], 1.0])  # np.roots 要從最高次排到常數
    print(f"AR({p})：最小的 |根| = {np.abs(roots).min():.3f}（> 1 才平穩）")

In [ ]:
# TODO 9（問答）實驗 1–3
#
# (1) 實驗 1：ACF、PACF 在 lag 1 與 lag 12 附近各是正還是負？哪一個在 lag 12（季節）之後比較快「截斷」？
#     依前置複習 5 的經驗法則，你會猜季節部分是 AR（P）還是 MA（Q）？（提示：經典的「航空模型」是 (0,1,1)(0,1,1)12）
# 答：
#
# (2) 實驗 2：
#     (a) AIC 最低的 p 是多少？p = 12、13 為什麼特別好？（提示：lag 12 = 去年同月）
#         程式裡每個 p 都只用同樣的 n − 14 個點算 AIC。如果直接用各自的 aic_（p 越大、用的點越少），會偏好哪一種 p？為什麼？
#     (b) 有常數 vs 沒有常數：test MASE 哪個好？兩次差分後的常數 c 代表什麼？（提示：z 的平均 ≠ 0 → x 每年多出一個二次的成長 —— 還原兩次差分時，常數會被「累加兩次」）
#     (c) 純 AR 模型最好的 test MASE 是多少？比「差分後預測 0」好嗎？比第 1 章的 0.66 呢？
# 答：
#
# (3) 實驗 3：所有的根都在單位圓外嗎？如果 OLS 估出不平穩的係數，遞迴預測會怎樣？
# 答：

---
## 4. 套件實作：statsmodels `SARIMAX`

`SARIMAX(endog, order=(p, d, q), seasonal_order=(P, D, Q, m))`：在**原始的 log y** 上指定差分次數，statsmodels 會自己差分、用**最大概似**（Kalman filter，第 5 章）估計 AR、MA 係數，預測時自己還原。

$$\underbrace{(1 - \phi_1 B - \cdots)}_{\text{AR}}\underbrace{(1 - \Phi_1 B^{12} - \cdots)}_{\text{季節 AR}}\underbrace{\nabla^d\nabla_{12}^D}_{\text{差分}} x_t = \underbrace{(1 + \theta_1 B + \cdots)}_{\text{MA}}\underbrace{(1 + \Theta_1 B^{12} + \cdots)}_{\text{季節 MA}}\varepsilon_t$$

（$B$ 是 backshift：$B x_t = x_{t-1}$。季節與非季節的多項式**相乘** → 例如 $(1 + \theta_1 B)(1 + \Theta_1 B^{12})$ 會產生 lag 13 的交叉項。）

| 模型 | 說明 |
| --- | --- |
| SARIMA($p$,1,0)(0,1,0)$_{12}$ | = 3E 的手寫 AR（沒有常數），只是估計方法不同（MLE vs OLS）→ 係數接近、不完全一樣 |
| SARIMA(0,1,1)(0,1,1)$_{12}$ | **航空模型**（Box & Jenkins 1970 用的就是這份資料） |
| AIC 網格 | $p, q \in \{0, 1, 2\}$、$P, Q \in \{0, 1\}$（$d = D = 1$），36 個模型，選 AIC 最低 |

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX


def fit_sarima(x: pd.Series, order: tuple, seasonal_order: tuple):
    return SARIMAX(x, order=order, seasonal_order=seasonal_order).fit(disp=False)


ar_sm = fit_sarima(x_train, (p_aic, 1, 0), (0, 1, 0, M))
ar_mine = ARScratch(p_aic, const=False).fit(z)
print(f"AR({p_aic})：手寫 OLS 前 3 個係數 {ar_mine.coef_[:3].round(4)}；SARIMAX MLE {ar_sm.params[:3].to_numpy().round(4)}")

airline = fit_sarima(x_train, (0, 1, 1), (0, 1, 1, M))
print("\n航空模型的參數：", airline.params.round(4).to_dict())

t0 = time.perf_counter()
grid = []
for p, q, P, Q in itertools.product(range(3), range(3), range(2), range(2)):
    res = fit_sarima(x_train, (p, 1, q), (P, 1, Q, M))
    grid.append({"order": (p, 1, q), "seasonal": (P, 1, Q), "AIC": res.aic, "BIC": res.bic})
grid = pd.DataFrame(grid).sort_values("AIC").reset_index(drop=True)
print(f"\n36 個模型共 {time.perf_counter() - t0:.0f} 秒；AIC 前 8 名：")
display(grid.head(8).round(2))
aic_best = fit_sarima(x_train, tuple(grid.loc[0, "order"]), tuple(grid.loc[0, "seasonal"]) + (M,))

pkg_rows = {}
for name, res in {f"SARIMA({p_aic},1,0)(0,1,0)": ar_sm, "airline (0,1,1)(0,1,1)": airline, f"AIC best {grid.loc[0, 'order']}{grid.loc[0, 'seasonal']}": aic_best}.items():
    pkg_rows[name] = {"AIC": res.aic, **forecast_metrics(y_test, np.exp(res.forecast(H)), y_train, season=M)}
display(pd.DataFrame(pkg_rows).T[["AIC", "mae", "rmse", "mase"]].round(3))

In [ ]:
# TODO 10（問答）SARIMAX
#
# (1) 手寫 OLS 的 AR 係數跟 SARIMAX 的 MLE 係數差多少？為什麼不完全一樣？（提示：OLS 丟掉前 p 個點、MLE 用全部的點；MLE 會考慮初始狀態）
# (2) 航空模型的 θ1、Θ1 是多少？都是負的 —— 跟實驗 1 的 ACF 在 lag 1、12 是負的一致嗎？
# (3) AIC 前幾名差多少？差 2 以內的模型通常被視為「差不多」。AIC 最低的模型，test 也最好嗎？
# 答：

---
## 5. 調參：滾動起點 CV（同第 1 章）

候選：季節性 naive、兩次差分的隨機漫步、手寫 AR($p$)（AIC 選的 $p$、沒有常數）、航空模型、AIC 網格最好的模型。CV 設定同第 1 章（起點 84、96、108，各預測 12 個月），全程只用 `y_train`。

`rolling_origin_cv` 是第 1 章 TODO 10 寫好的版本，搬過來（已完成）。

In [ ]:
def rolling_origin_cv(y: pd.Series, make_forecast, origins: list[int], horizon: int = 12, m: int = M) -> pd.DataFrame:
    """第 1 章的版本：make_forecast(train_series, horizon) → (horizon,) 的預測。"""
    rows = []
    for origin in origins:
        tr = y.iloc[:origin]
        va = y.iloc[origin : origin + horizon]
        fc = make_forecast(tr, horizon)
        rows.append({"origin": str(y.index[origin].date()), **forecast_metrics(va, fc, tr, season=m)})
    return pd.DataFrame(rows).set_index("origin")


def ar_forecaster(p: int):
    def f(tr: pd.Series, h: int) -> np.ndarray:
        xt = np.log(tr).to_numpy()
        zt = np.log(tr).diff().diff(M).dropna().to_numpy()
        return np.exp(undifference(xt, ARScratch(p, const=False).fit(zt).forecast(h)))
    return f


def sarima_forecaster(order: tuple, seasonal: tuple):
    return lambda tr, h: np.exp(fit_sarima(np.log(tr), order, seasonal + (M,)).forecast(h).to_numpy())


ORIGINS = [84, 96, 108]
best_order, best_seas = tuple(grid.loc[0, "order"]), tuple(grid.loc[0, "seasonal"])
candidates = {
    "snaive": lambda tr, h: tr.to_numpy()[len(tr) - M + (np.arange(h) % M)],
    "∇∇12 random walk": lambda tr, h: np.exp(undifference(np.log(tr).to_numpy(), np.zeros(h))),
    f"AR({p_aic}) scratch": ar_forecaster(p_aic),
    "airline (0,1,1)(0,1,1)": sarima_forecaster((0, 1, 1), (0, 1, 1)),
    f"AIC best {best_order}{best_seas}": sarima_forecaster(best_order, best_seas),
}
N_PARAMS = {"snaive": 0, "∇∇12 random walk": 0, f"AR({p_aic}) scratch": p_aic, "airline (0,1,1)(0,1,1)": 2, f"AIC best {best_order}{best_seas}": sum(best_order) - 1 + sum(best_seas) - 1}
cv_rows = {}
for name, f in candidates.items():
    cv = rolling_origin_cv(y_train, f, ORIGINS)
    cv_rows[name] = {"mean MASE": cv["mase"].mean(), "std MASE": cv["mase"].std(), **{f"MASE @{o}": v for o, v in cv["mase"].items()}}
cv_table = pd.DataFrame(cv_rows).T.sort_values("mean MASE")
display(cv_table.round(3))
best = cv_table.index[0]
se_best = cv_table.loc[best, "std MASE"] / np.sqrt(len(ORIGINS))
close = cv_table[cv_table["mean MASE"] <= cv_table.loc[best, "mean MASE"] + se_best].index.tolist()
chosen = min(close, key=lambda k: (N_PARAMS[k], cv_table.loc[k, "mean MASE"]))
print(f"CV 最好：{best}；一個標準誤內：{close} → 選 {chosen}（參數最少）")

In [ ]:
# TODO 11（問答）調參
#
# (1) CV 的排名跟 AIC 的排名一致嗎？為什麼可能不一致？（AIC：一步預測的配適 + 參數懲罰；CV：1～12 步的預測）
# (2) 選出的模型是哪一個？如果有幾個模型在一個標準誤以內，選參數最少的理由是什麼？
# (3) 只有 3 個 fold 時，「一個標準誤」有多大？如果 one-standard-error rule 選到了一個很簡單、但 test 上很差的模型，這說明什麼？
#     （提示：fold 太少 → 標準誤很大 → 幾乎所有模型都「分不出差異」；第 1 章 TODO 11 (3) 的改進方法）
# 答：

---
## 6. 測試集評估

用選出的模型在完整的 `y_train` 上估計、預測 test 24 個月，只評估一次。參照點：季節性 naive、航空模型、以及第 1 章最好的模型（log + Holt-Winters 加法，這裡用 statsmodels 的 `ExponentialSmoothing` 重算一次 —— statsmodels 的初始狀態與參數估計方式跟第 1 章的手寫版不同，成績也會不同；第 1 章手寫版的 test MASE 是 0.66）。成績接著寫進第 1 章的 `results/air_passengers.csv`。

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from ml_dl_llm.results import compare, log_result

BENCH = "air_passengers"
fc_chosen = candidates[chosen](y_train, H)
fc_es = np.exp(ExponentialSmoothing(x_train, trend="add", seasonal="add", seasonal_periods=M).fit().forecast(H).to_numpy())
eval_models = {
    f"{chosen} (CV-selected)": fc_chosen,
    "airline SARIMA(0,1,1)(0,1,1)12 on log": candidates["airline (0,1,1)(0,1,1)"](y_train, H),
    f"AR({p_aic}) on ∇∇12 log y (scratch)": candidates[f"AR({p_aic}) scratch"](y_train, H),
    "∇∇12 random walk": fc_rw,
    "(ch.1) log + HW additive, statsmodels": fc_es,
}
for name, fc in eval_models.items():
    test_table.loc[name] = forecast_metrics(y_test, fc, y_train, season=M)
    if not name.startswith("(ch.1)"):
        log_result(BENCH, name, test_table.loc[name].dropna().to_dict(), horizon=H)
display(test_table.round(4))
compare(BENCH, "mase", higher_is_better=False)

---
## 7. 診斷

1. **殘差**（航空模型、AIC 最好的模型）：ACF、Ljung-Box、常態性（Q-Q plot）。
2. **預測區間**：SARIMAX 的 `get_forecast(h).conf_int()` 在 log 尺度上是常態 → exp 回來是不對稱的區間。95% 區間涵蓋了 test 的幾 %？
3. **哪一段預測錯最多**：test 每個月的誤差（log 尺度的百分比誤差）。

In [ ]:
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox

fig, axes = plt.subplots(1, 3, figsize=(18, 3.6))
res_air = airline.resid.iloc[M + 1 :]  # 去掉差分吃掉的前 13 期
axes[0].bar(np.arange(1, 25), acf(res_air.to_numpy(), 24)[1:])
axes[0].axhline(2 / np.sqrt(len(res_air)), color="gray", ls=":")
axes[0].axhline(-2 / np.sqrt(len(res_air)), color="gray", ls=":")
axes[0].set_title("airline residual ACF")
stats.probplot(res_air, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot")
fcst = airline.get_forecast(H)
ci = np.exp(fcst.conf_int(alpha=0.05).to_numpy())
axes[2].plot(y.iloc[-48:].index, y.iloc[-48:].to_numpy(), color="k", label="actual")
axes[2].plot(y_test.index, np.exp(fcst.predicted_mean.to_numpy()), color="tab:red", label="airline")
axes[2].plot(y_test.index, fc_es, color="tab:green", ls="--", label="ch.1 log HW")
axes[2].fill_between(y_test.index, ci[:, 0], ci[:, 1], alpha=0.2, color="tab:red", label="95% PI")
axes[2].legend(fontsize=8)
plt.show()
display(acorr_ljungbox(res_air, lags=[6, 12, 24]).round(4))
print(f"Shapiro-Wilk 常態檢定 p 值：{stats.shapiro(res_air).pvalue:.3f}；95% 預測區間涵蓋 test 的 {np.mean((y_test.to_numpy() >= ci[:, 0]) & (y_test.to_numpy() <= ci[:, 1])):.0%}")
pct_err = pd.DataFrame({"airline": 100 * (np.exp(fcst.predicted_mean.to_numpy()) / y_test.to_numpy() - 1), "ch.1 log HW": 100 * (fc_es / y_test.to_numpy() - 1)}, index=y_test.index.strftime("%Y-%m"))
display(pct_err.round(1).T)

In [ ]:
# TODO 12（問答，本章總結）
#
# (1) 測試集：依 MASE 排序本章的模型與第 1 章的模型。ARIMA 有贏過指數平滑嗎？差多少？
# (2) 殘差：Ljung-Box 的 p 值、Q-Q plot 的尾巴 —— 航空模型的假設（白雜訊、常態）大致成立嗎？
# (3) 預測區間涵蓋了 test 的幾 %？百分比誤差表：兩個模型都系統性地高估還是低估？集中在哪幾個月？
#     （提示：1960 年的成長比過去慢 —— 任何只看過去的模型都會「延續過去的成長」）
# (4) 指數平滑 vs ARIMA：
#     |                 | 指數平滑 | ARIMA |
#     | 怎麼處理趨勢    |          |       |
#     | 怎麼處理季節    |          |       |
#     | 參數怎麼估      |          |       |
#     | 怎麼選模型      |          |       |
#     | 本章的 test MASE |         |       |
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

AirPassengers（train 1949–1958、test 1959–1960，24 步預測）

| 方法 | 做法 | AIC | CV mean MASE | test MASE |
| --- | --- | --- | --- | --- |
| 季節性 naive | | — | | |
| ∇∇12 隨機漫步 | | — | | |
| AR($p$) 手寫（OLS） | | | | |
| 航空模型 SARIMA(0,1,1)(0,1,1)$_{12}$ | | | | |
| AIC 網格最好 | | | | |
| （第 1 章）log + Holt-Winters 加法 | | — | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ MA 與 ARMA 的最大概似估計（Kalman filter 的狀態空間形式 → 第 5 章）
- ☐ 單位根檢定的細節（ADF、KPSS、季節單位根）；過度差分的問題
- ☐ ARIMA 預測區間的推導（ψ 權重）
- ☐ 指數平滑與 ARIMA 的等價：SES = ARIMA(0,1,1)、加法 Holt-Winters ≈ SARIMA

**延伸**
- ☐ Auto-ARIMA（`pmdarima`、`statsforecast` 的 AutoARIMA）：逐步搜尋
- ☐ ARIMAX：加上外生變數（節日、促銷）
- ☐ Box-Cox 轉換（log 是 λ = 0 的特例）